# SIA Companion — LoRA Fine-Tune (Gemma 3 1B)

Trains SIA's chat personality onto a small instruct base model that fits on **free Colab T4** (and your 8GB MacBook for inference).

- Base: `google/gemma-3-1b-it` (or swap to `LFM2.5 1.2B` if you prefer)
- Method: LoRA (r=16, alpha=16) via PEFT + TRL SFTTrainer, response-only loss
- Output: adapter in `sia-lora/` + optional merged GGUF-ready model

**Run order:** Cell 1 → 2 → 3 → 4 (train) → 5 (test chat). ~15–25 min on T4.

In [ ]:
# Cell 1 — installs
!pip install -q "transformers>=4.44" peft trl datasets accelerate bitsandbytes

In [ ]:
# Cell 2 — dataset: paste your sia_chat_train.jsonl here (or mount Drive)
import json
from datasets import Dataset

# Option A: upload sia_chat_train.jsonl to this Colab session (files panel)
# Option B: mount Drive and point here
PATH = "sia_chat_train.jsonl"

rows = [json.loads(l) for l in open(PATH, encoding="utf-8")]
ds = Dataset.from_list(rows)
print("samples:", len(ds))
print(ds[0]["messages"])

In [ ]:
# Cell 3 — model + LoRA config (T4-friendly)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

BASE = "google/gemma-3-1b-it"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16
)

lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# Cell 4 — SFT trainer (response-only loss via chat template)
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir="sia-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=30,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="epoch",
    max_seq_length=512,
    packing=False,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=ds,
    tokenizer=tok,
    dataset_text_field="messages",
    processing_class=tok,  # TRL >= 0.12: processing_class replaces tokenizer kwarg
)
trainer.train()
trainer.save_model("sia-lora-final")
print("saved adapter -> sia-lora-final/")

In [ ]:
# Cell 5 — test the fine-tuned SIA
from peft import PeftModel

model = PeftModel.from_pretrained(model, "sia-lora-final")
model.eval()

def chat(msg: str):
    ids = tok.apply_chat_template(
        [{"role": "user", "content": msg}], add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(ids, max_new_tokens=128, do_sample=True, temperature=0.7, top_k=40)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

print("SIA:", chat("Namaste! What can you do?"))
print("SIA:", chat("क्या तुम ऑफलाइन काम कर सकते हो?"))

## After training
- Download `sia-lora-final/` (adapter ~30MB).
- Merge + export to GGUF (llama.cpp) for 8GB-device inference: see `finetune/merge_and_export.py` in the repo.
- Push adapter to Hugging Face: `!huggingface-cli upload sia-lora-final sia-lora --repo-type model`